# gradb2: which sparkle version is in each global store

One figure. Reads `gradb2` from both global datasets and computes it live on the
Monterey chunk, all at the same place and time, in a 200 km box.

The live tile is the reference. `dbof.preprocessing.calculate_fields.grad_b2` is
called directly on the chunk — the same function `surface_subsets` maps
`"gradb2"` to, so it is the pipeline's own code on whatever `dbof` is installed
now. Nothing is written to disk. Whichever store matches it is current.

## The sparkle

At a one-cell gradient extremum the slopes on either side are `-a` and `+a`.
Interpolating them to the cell centre averages them to **zero**, and squaring
locks that in. The old form therefore *manufactures near-zeros* — holes — at
exactly the grid-scale features that matter. The fix squares on the staggered
points first, keeping the real value.

Per direction the two forms differ by an exact identity:

```
new - old  =  1/4 (g1 - g2)^2
```

so the difference is **never negative**, and is a map of grid-scale gradient
curvature: zero in smooth water, bright along filaments. Blue in a
`new - old` panel would mean something other than the formula is going on.

## Provenance

| | |
|---|---|
| fix committed | `88fcf1d` 2026-08-06 23:04 -0700 |
| merged to `main` | `a3e43eb` 2026-08-10 07:52 -0700 (PR #31 `gradient-sparkles`) |
| `globals_for_chunks/V5` | current form — matches the live tile |
| `globals_for_cutouts/v2_2_01` | superseded form; chunks written 2026-08-10 18:16 UTC |

`v2_2_01/run_meta.yaml` records `git_commit: 48f5031` (2026-08-10 07:58 -0700),
which *does* contain the fix — but `_git_commit_hash()` in
`dbof.global_dataset_creation.metadata` runs `git rev-parse HEAD` with no `cwd`
and no reference to `dbof.__file__`. It records the repo the shell was standing
in, not the package that was imported, so it cannot testify to what ran. The
measurement below is the evidence, not the metadata.

Channels affected by the fix: `gradb2`, `gradtheta2`, `gradsalt2`, `gradrho2`,
`gradeta2`, `strain_mag`, `okubo_weiss`. Not `relative_vorticity`,
`divergence`, `rossby_number`, `turner_angle`, `density`, `buoyancy`,
`frontogenesis_tendency`.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from fronts.llc import io as llc_io
from fronts.llc import tiles as llc_tiles

# The pipeline's own function: surface_subsets maps "gradb2" -> grad_b2.
from dbof.preprocessing import calculate_fields as cf

CFG_CHUNKS  = '../../runs/prototypes/one_full/run_v5_chunks.yaml'
CFG_CUTOUTS = '../../runs/prototypes/one_full/run_v5_100_timesteps.yaml'
CHUNK       = 'monterey_bay'
DATE        = '2012-07-04T12_00_00'      # in both configs
BOX_KM      = 200

# Sequential: one hue light->dark for magnitude.  Diverging: blue<->red with a
# neutral gray midpoint so "no difference" reads as nothing.
SEQ = 'Blues'
DIV = LinearSegmentedColormap.from_list(
    'blue_gray_red', ['#184f95', '#86b6ef', '#f0efec', '#f0a3a2', '#a52322'])

# Jupyter normally picks the inline backend up from the magic above.  When it
# does not, matplotlib stays on Agg, which draws to a buffer and renders
# nothing -- plt.show() becomes a silent no-op.  Force it.
import matplotlib
if 'inline' not in matplotlib.get_backend():
    try:
        plt.switch_backend('module://matplotlib_inline.backend_inline')
    except Exception as exc:
        print(f'could not switch off {matplotlib.get_backend()}: {exc}\n'
              'figures will not render -- pip install matplotlib-inline '
              'and restart the kernel')
print('matplotlib backend:', matplotlib.get_backend())

In [ ]:
tile = llc_tiles.tile_from_chunk_store(CHUNK)

# The chunk store IS the tile, so nothing is sliced.  chunk_context opens the
# snapshot and its grid and builds the xgcm grid over them.
ds_merge, xgrid = llc_tiles.chunk_context(CHUNK, DATE)

# Box size from the store's own metric, so it really is BOX_KM across.
dx_km = float(np.nanmean(ds_merge['dxC'].values)) / 1e3
half = int(BOX_KM / dx_km / 2)
c = llc_tiles.TILE_SIZE // 2
box = (slice(c - half, c + half), slice(c - half, c + half))

fields = {}
for label, cfg_file, rid in (('globals_for_chunks/V5', CFG_CHUNKS, 'V5'),
                             ('globals_for_cutouts/v2_2_01', CFG_CUTOUTS, 'v2_2_01')):
    g = llc_io.read_channel(cfg_file, DATE, 'frontal_structure', 'gradb2',
                            run_id=rid)
    fields[label] = llc_tiles.labels_for_tile(g, tile)[box]
    del g

# Live: grad_b2 applied to the chunk on the spot.  Nothing is written.
fields['tile, computed live'] = llc_tiles.surface(cf.grad_b2(ds_merge, xgrid))[box]

ref = 'tile, computed live'
pairs = [(k, ref) for k in fields if k != ref]
pairs.append(('globals_for_chunks/V5', 'globals_for_cutouts/v2_2_01'))

print(f'{2*half} x {2*half} cells at dxC = {dx_km:.2f} km '
      f'= {2*half*dx_km:.0f} km across\n')
for a, b in pairs:
    A, B = fields[a], fields[b]
    m = np.isfinite(A) & np.isfinite(B) & (B > 0)
    rel = np.abs(A[m] - B[m]) / B[m]
    print(f'{a} vs {b}\n    median |rel| {np.median(rel):.2e}   '
          f'p95 {np.percentile(rel, 95):.2e}')

In [ ]:
logs  = {k: np.log10(np.where(v > 0, v, np.nan)) for k, v in fields.items()}
diffs = {f'{a}\n$-$ {b}': logs[a] - logs[b] for a, b in pairs}
vmin, vmax = np.nanpercentile(logs[ref], [2, 98])
lim = max(float(np.nanpercentile(np.abs(d), 99)) for d in diffs.values())

fig, ax = plt.subplots(2, 3, figsize=(16, 9.2))
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
    for s in a.spines.values():
        s.set_color('#b8b6b0')

for col, (name, arr) in enumerate(logs.items()):
    im = ax[0, col].imshow(arr, origin='lower', cmap=SEQ, vmin=vmin, vmax=vmax)
    ax[0, col].set_title(name, fontsize=10)
    fig.colorbar(im, ax=ax[0, col], fraction=0.046, label='log$_{10}$ gradb2')

for col, (name, arr) in enumerate(diffs.items()):
    im = ax[1, col].imshow(arr, origin='lower', cmap=DIV, vmin=-lim, vmax=lim)
    ax[1, col].set_title(name, fontsize=10)
    fig.colorbar(im, ax=ax[1, col], fraction=0.046, label='dex')

fig.suptitle(f'gradb2  {DATE}  {CHUNK}  —  {2*half*dx_km:.0f} km box',
             fontsize=12)
fig.tight_layout()
plt.show()